### Imports

In [12]:
import requests
import pandas as pd
from io import BytesIO
from datetime import datetime, timezone
from metar import Metar

### Today's Date in UTC

In [13]:
today = datetime.now(timezone.utc)

### METAR Data

In [ ]:
def fetch_metar(start_date, end_date, station):
    """Fetch and parse historical METAR observations for an airport from IEM (Iowa Environmental Mesonet).

    Args:
    - start_date (str | datetime): inclusive start date as 'MM/DD/YYYY' string or datetime.
    - end_date (str | datetime): exclusive end date in the same format.
    - station (str): full ICAO airport code (e.g., 'KMDW').

    Returns:
    - pd.DataFrame: one row per METAR (routine) or SPECI (special) observation, with columns:
        - station (str): ICAO airport code, mirrors the input.
        - timestamp (datetime64[ns, UTC]): observation time.
        - report_type (str): 'METAR' (routine) or 'SPECI' (special).
        - mod (str): 'AUTO' (automated) or 'COR' (corrected).
        - wdir (Int64): wind direction in degrees true.
        - wspd_kt (Int64): wind speed in knots.
        - gust_kt (Int64): wind gust in knots.
        - wdir_from / wdir_to (Int64): variable wind direction range, degrees.
        - peak_wspd_kt (Int64): peak wind speed in the last hour, knots.
        - peak_wdir (Int64): peak wind direction in the last hour, degrees.
        - vis_sm (float): prevailing visibility in statute miles (fractional, e.g. 0.25 for 1/4SM).
        - wx (str): present weather codes as a standard METAR string (e.g., '-RA BR').
        - sky (str): sky cover layers as a standard METAR string (e.g., 'BKN020CB OVC060').
        - temp_c (float): temperature in °C (0.1° resolution when T-group remarks are present, whole °C otherwise).
        - dewp_c (float): dewpoint in °C.
        - max6_c / min6_c (float): 6-hour max/min temperature in °C.
        - max24_c / min24_c (float): 24-hour max/min temperature in °C.
        - press_mb (float): altimeter setting in millibars.
        - slp_mb (float): sea-level pressure in millibars.
        - p1hr_in / p3hr_in / p6hr_in / p24hr_in (float): precipitation totals in inches.
        - snow_in (Int64): snow depth in whole inches.
    """
    start = pd.to_datetime(start_date, format='%m/%d/%Y')
    end = pd.to_datetime(end_date, format='%m/%d/%Y')
    url = (
        "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"
        f"?station={station}&data=metar"
        f"&year1={start.year}&month1={start.month}&day1={start.day}"
        f"&year2={end.year}&month2={end.month}&day2={end.day}"
        "&tz=UTC&format=onlycomma&missing=M"
        "&report_type=3&report_type=4"
    )
    raw_metar = pd.read_csv(BytesIO(requests.get(url).content))
    parsed_metar = raw_metar["metar"].apply(_parse_metar).apply(pd.Series)
    parsed_metar.insert(0, 'timestamp', pd.to_datetime(raw_metar['valid'], utc=True))
    parsed_metar.insert(0, 'station', station)
    for c in ["wdir", "wspd_kt", "gust_kt", "wdir_from", "wdir_to",
              "peak_wspd_kt", "peak_wdir", "snow_in"]:
        parsed_metar[c] = parsed_metar[c].astype("Int64")
    return parsed_metar



def _parse_metar(raw_metar):
    """Decode a single raw METAR/SPECI report string into a flat dict of fields.

    Uses python-metar to parse, then extracts a curated subset of attributes. Weather (`wx`)
    and sky (`sky`) groups are re-encoded as standard METAR strings so the dict serializes
    cleanly to parquet/CSV. Numeric fields are returned as float; the caller (fetch_metar)
    is responsible for casting whole-integer columns to Int64.

    Args:
    - raw_metar (str): a single METAR or SPECI report string.

    Returns:
    - dict: decoded fields keyed by the short column names listed in fetch_metar.
      Returns an empty dict on any parse failure, so the caller's DataFrame build
      produces NaNs for the missing keys.
    """
    try:
        obs = Metar.Metar(raw_metar)
        parsed_metar = {
            # identity / metadata
            "report_type":  obs.type,
            "mod":          obs.mod,
            # wind
            "wdir":         obs.wind_dir.value() if obs.wind_dir else None,
            "wspd_kt":      obs.wind_speed.value("KT") if obs.wind_speed else None,
            "gust_kt":      obs.wind_gust.value("KT") if obs.wind_gust else None,
            "wdir_from":    obs.wind_dir_from.value() if obs.wind_dir_from else None,
            "wdir_to":      obs.wind_dir_to.value() if obs.wind_dir_to else None,
            "peak_wspd_kt": obs.wind_speed_peak.value("KT") if obs.wind_speed_peak else None,
            "peak_wdir":    obs.wind_dir_peak.value() if obs.wind_dir_peak else None,
            # visibility & weather (TAF-aligned, flattened to standard METAR strings)
            "vis_sm":       obs.vis.value("SM") if obs.vis else None,
            "wx":           " ".join("".join(p or "" for p in tup) for tup in obs.weather) if obs.weather else None,
            "sky":          " ".join(
                                f"{c}{int(h.value('FT'))//100:03d}{t or ''}" if h else f"{c}{t or ''}"
                                for c, h, t in obs.sky
                            ) if obs.sky else None,
            # thermodynamics
            "temp_c":       obs.temp.value("C") if obs.temp else None,
            "dewp_c":       obs.dewpt.value("C") if obs.dewpt else None,
            "max6_c":       obs.max_temp_6hr.value("C") if obs.max_temp_6hr else None,
            "min6_c":       obs.min_temp_6hr.value("C") if obs.min_temp_6hr else None,
            "max24_c":      obs.max_temp_24hr.value("C") if obs.max_temp_24hr else None,
            "min24_c":      obs.min_temp_24hr.value("C") if obs.min_temp_24hr else None,
            # pressure
            "press_mb":     obs.press.value("MB") if obs.press else None,
            "slp_mb":       obs.press_sea_level.value("MB") if obs.press_sea_level else None,
            # precip / snow
            "p1hr_in":      obs.precip_1hr.value("IN") if obs.precip_1hr else None,
            "p3hr_in":      obs.precip_3hr.value("IN") if obs.precip_3hr else None,
            "p6hr_in":      obs.precip_6hr.value("IN") if obs.precip_6hr else None,
            "p24hr_in":     obs.precip_24hr.value("IN") if obs.precip_24hr else None,
            "snow_in":      obs.snowdepth.value("IN") if obs.snowdepth else None,
        }
        return parsed_metar
    except Exception:
        return {}

In [15]:
metar = fetch_metar('01/01/2020', today, "KMDW")

In [16]:
metar.head()

,station,timestamp,report_type,mod,wdir,wspd_kt,gust_kt,wdir_from,wdir_to,peak_wspd_kt,...,min6_c,max24_c,min24_c,press_mb,slp_mb,p1hr_in,p3hr_in,p6hr_in,p24hr_in,snow_in
0,KMDW,2020-01-01 00:53:00+00:00,METAR,AUTO,280,13,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,1009.823884,1011.0,NaN,NaN,NaN,NaN,<NA>
1,KMDW,2020-01-01 01:53:00+00:00,METAR,AUTO,270,11,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,1010.162523,1011.3,NaN,NaN,NaN,NaN,<NA>
2,KMDW,2020-01-01 02:29:00+00:00,METAR,AUTO,270,10,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,1010.162523,NaN,NaN,NaN,NaN,NaN,<NA>
3,KMDW,2020-01-01 02:53:00+00:00,METAR,AUTO,260,15,23,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,1009.823884,1011.1,NaN,NaN,NaN,NaN,<NA>
4,KMDW,2020-01-01 03:37:00+00:00,METAR,AUTO,260,10,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,1009.823884,NaN,NaN,NaN,NaN,NaN,<NA>


### TAF Data

In [17]:
def fetch_taf(start_iso, end_iso, station):
    """Fetch the parsed TAF (Terminal Aerodrome Forecast) history for an airport from IEM.

    Each row is one forecast segment within a TAF issuance: one 'Observation' row per
    issuance (initial-conditions snapshot) plus one 'Forecast' row per FM/BECMG/TEMPO
    change group, already decoded by IEM.

    Args:
    - start_iso (str): inclusive start time as ISO string (e.g., '2020-01-01T00:00Z').
    - end_iso (str): exclusive end time in the same format.
    - station (str): bare 3-letter station suffix (e.g., 'MDW'). The 'K' is added in
      the URL (US-only).

    Returns:
    - pd.DataFrame: TAF forecast segments with columns:
        - station (str): full ICAO airport code.
        - valid (datetime64[ns, UTC]): TAF issuance time.
        - fx_valid (datetime64[ns, UTC]): forecast period start.
        - fx_valid_end (datetime64[ns, UTC]): forecast period end (NaT for FM groups, open-ended).
        - raw (str): raw TAF segment text (e.g., 'FM011700 21014G24KT P6SM SCT250').
        - is_tempo (bool): True if this is a TEMPO group (temporary fluctuation).
        - sknt (Int64): forecast wind speed in knots.
        - drct (Int64): forecast wind direction in degrees true.
        - gust (Int64): forecast gust in knots (0 if none).
        - visibility (float): forecast visibility in statute miles (6.01 encodes 'P6SM').
        - presentwx (str): list of forecast weather phenomena, serialized as a string.
        - skyc (str): list of sky cover codes, serialized as a string.
        - skyl (str): list of layer heights in feet AGL, serialized as a string.
        - ws_level (Int64): non-convective wind shear altitude in ft AGL.
        - ws_drct (Int64): wind shear direction in degrees true.
        - ws_sknt (Int64): wind shear speed in knots.
        - product_id (str): NWS product header (issuance time + WFO + AWIPS id).
        - ftype (str): 'Observation' or 'Forecast'.
        - is_amendment (object): True if the TAF was AMD/COR; dtype is mixed in source CSV.
    """
    url = (
        "https://mesonet.agron.iastate.edu/cgi-bin/request/taf.py"
        f"?station=K{station}&sts={start_iso}&ets={end_iso}&fmt=csv"
    )
    taf = pd.read_csv(BytesIO(requests.get(url).content), low_memory=False)
    for c in ["valid", "fx_valid", "fx_valid_end"]:
        taf[c] = pd.to_datetime(taf[c], utc=True)
    for c in ["sknt", "drct", "gust", "ws_level", "ws_drct", "ws_sknt"]:
        taf[c] = taf[c].astype("Int64")
    return taf

In [18]:
taf = fetch_taf("2020-01-01T00:00Z", today.strftime("%Y-%m-%dT%H:%MZ"), "MDW")

In [19]:
taf.head()

,station,valid,fx_valid,raw,is_tempo,fx_valid_end,sknt,drct,gust,visibility,presentwx,skyc,skyl,ws_level,ws_drct,ws_sknt,product_id,ftype,is_amendment
0,KMDW,2020-01-01 03:02:00+00:00,2020-01-01 03:02:00+00:00,26013KT P6SM FEW020,False,NaT,13,260,0,6.01,[],['FEW'],[2000],<NA>,<NA>,<NA>,202001010302-KLOT-FTUS43-TAFMDW-AAA,Observation,True
1,KMDW,2020-01-01 03:02:00+00:00,2020-01-01 17:00:00+00:00,FM011700 21014G24KT P6SM SCT250,False,NaT,14,210,24,6.01,[],['SCT'],[25000],<NA>,<NA>,<NA>,202001010302-KLOT-FTUS43-TAFMDW-AAA,Forecast,True
2,KMDW,2020-01-01 03:02:00+00:00,2020-01-01 23:00:00+00:00,FM012300 20014KT P6SM SCT180,False,NaT,14,200,0,6.01,[],['SCT'],[18000],<NA>,<NA>,<NA>,202001010302-KLOT-FTUS43-TAFMDW-AAA,Forecast,True
3,KMDW,2020-01-01 05:33:00+00:00,2020-01-01 05:33:00+00:00,25009KT P6SM SCT020,False,NaT,9,250,0,6.01,[],['SCT'],[2000],<NA>,<NA>,<NA>,202001010533-KLOT-FTUS43-TAFMDW,Observation,False
4,KMDW,2020-01-01 05:33:00+00:00,2020-01-01 17:00:00+00:00,FM011700 21014G24KT P6SM SCT250,False,NaT,14,210,24,6.01,[],['SCT'],[25000],<NA>,<NA>,<NA>,202001010533-KLOT-FTUS43-TAFMDW,Forecast,False


### ASOS Data

In [ ]:
def fetch_asos_1min(start_date, end_date, station):
    """Fetch 1-minute ASOS observations for an airport from IEM.

    Higher cadence than METAR with a smaller field set, natively reported in Fahrenheit
    and whole-knot/degree units. Useful as a high-resolution truth source for minute-level
    analysis or for matching NWS CLI daily reports.

    Args:
    - start_date (str | datetime): inclusive start date as 'MM/DD/YYYY' string or datetime.
    - end_date (str | datetime): exclusive end date in the same format.
    - station (str): full ICAO airport code (e.g., 'KMDW'). The 'K' is stripped
      internally before being sent to IEM's asos1min endpoint, which uses 3-letter
      FAA identifiers.

    Returns:
    - pd.DataFrame: ASOS 1-minute observations with columns:
        - station (str): full ICAO airport code, mirrors the input.
        - timestamp (datetime64[ns, UTC]): observation time.
        - temp_f (Int64): temperature in whole °F.
        - dewp_f (Int64): dewpoint in whole °F.
        - wdir (Int64): wind direction in degrees true.
        - wspd_kt (Int64): wind speed in knots.
        - gust_wdir (Int64): gust direction in degrees true.
        - gust_kt (Int64): gust speed in knots.
        - ptype (str): precipitation type code (e.g., 'NP' = no precipitation).
        - precip_in (float): 1-minute precipitation accumulation in inches (0.01 in resolution).
    """
    start = pd.to_datetime(start_date, format='%m/%d/%Y')
    end = pd.to_datetime(end_date, format='%m/%d/%Y')
    bare = station[1:] if station.startswith("K") else station
    url = (
        "https://mesonet.agron.iastate.edu/cgi-bin/request/asos1min.py"
        f"?station[]={bare}"
        f"&year1={start.year}&month1={start.month}&day1={start.day}&hour1=0&minute1=0"
        f"&year2={end.year}&month2={end.month}&day2={end.day}&hour2=0&minute2=0"
        "&vars[]=tmpf&vars[]=dwpf&vars[]=sknt&vars[]=drct"
        "&vars[]=gust_sknt&vars[]=gust_drct&vars[]=ptype&vars[]=precip"
        "&sample=1min&what=download&tz=UTC"
    )
    raw_asos = pd.read_csv(BytesIO(requests.get(url).content), na_values=['M'], low_memory=False)
    asos = pd.DataFrame({
        "station":   station,
        "timestamp": pd.to_datetime(raw_asos["valid(UTC)"], utc=True),
        "temp_f":    raw_asos["tmpf"].astype("Int64"),
        "dewp_f":    raw_asos["dwpf"].astype("Int64"),
        "wdir":      raw_asos["drct"].astype("Int64"),
        "wspd_kt":   raw_asos["sknt"].astype("Int64"),
        "gust_wdir": raw_asos["gust_drct"].astype("Int64"),
        "gust_kt":   raw_asos["gust_sknt"].astype("Int64"),
        "ptype":     raw_asos["ptype"],
        "precip_in": raw_asos["precip"],
    })
    return asos

In [21]:
asos = fetch_asos_1min('01/01/2020', today, "KMDW")

In [22]:
asos.head()

,station,timestamp,temp_f,dewp_f,wdir,wspd_kt,gust_wdir,gust_kt,ptype,precip_in
0,KMDW,2020-01-01 00:00:00+00:00,29,21,272,14,282,19,NP,0.0
1,KMDW,2020-01-01 00:01:00+00:00,29,21,279,14,280,19,NP,0.0
2,KMDW,2020-01-01 00:02:00+00:00,29,21,280,15,272,19,NP,0.0
3,KMDW,2020-01-01 00:03:00+00:00,29,21,278,14,274,17,NP,0.0
4,KMDW,2020-01-01 00:04:00+00:00,29,21,273,13,260,17,NP,0.0


### NWS CLI Climatological Reports

In [25]:
def fetch_cli(start_date, end_date, station):
    """Fetch daily NWS CLI (Climatological Daily Report) summaries from IEM (Iowa Environmental Mesonet).

    CLI products are issued daily by the local NWS WFO and summarize the previous
    climate day (local time, midnight to midnight). IEM serves the pre-parsed CLI
    fields as JSON one year at a time.

    Args:
    - start_date (str | datetime): inclusive start date as 'MM/DD/YYYY' string or datetime.
    - end_date (str | datetime): exclusive end date in the same format.
    - station (str): full ICAO airport code (e.g., 'KMDW').

    Returns:
    - pd.DataFrame: one row per calendar day, with columns:
        - station (str): ICAO airport code, mirrors the input.
        - date (datetime64[ns]): calendar date the CLI report covers (local-time climate day, naive).
        - max_temp_f (Int64): daily maximum temperature in whole °F.
    """
    start = pd.to_datetime(start_date, format='%m/%d/%Y')
    end = pd.to_datetime(end_date, format='%m/%d/%Y')
    frames = []
    for year in range(start.year, end.year + 1):
        url = f"https://mesonet.agron.iastate.edu/json/cli.py?station={station}&year={year}"
        frames.append(pd.DataFrame(requests.get(url).json()["results"]))
    raw = pd.concat(frames, ignore_index=True)
    cli = pd.DataFrame({
        "station":    station,
        "date":       pd.to_datetime(raw["valid"]),
        "max_temp_f": pd.to_numeric(raw["high"], errors="coerce").astype("Int64"),
    })
    return cli[(cli["date"].dt.date >= start.date()) & (cli["date"].dt.date < end.date())].reset_index(drop=True)

In [26]:
cli = fetch_cli('01/01/2020', today, "KMDW")

In [28]:
cli.tail()

,station,date,max_temp_f
2324,KMDW,2026-05-13,59
2325,KMDW,2026-05-14,65
2326,KMDW,2026-05-15,76
2327,KMDW,2026-05-16,83
2328,KMDW,2026-05-17,87
